## Pulling the Data

In [1]:
# Imports
import oracledb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import altair as alt

In [20]:
# Connecting to our database

dsn = oracledb.makedsn("localhost", 1522, service_name="stu")
connection = oracledb.connect(user="ora_inigogrc", password="a77155190", dsn=dsn)

In [21]:
# # Inspecting our parent table

# cur = connection.cursor()
# for row in cur.execute("SELECT * FROM parent_table WHERE ROWNUM <= 5"):
#     print(row)
# cur.close()

In [26]:
# Inspecting our parent table
cur = connection.cursor()
cur.execute("SELECT * FROM parent_table WHERE ROWNUM <= 5")
rows = cur.fetchall()  # fetch all rows as a list of tuples

# Extract column names from the cursor description
col_names = [desc[0] for desc in cur.description]
cur.close()

# Create a DataFrame with the rows and the column names
parent_df = pd.DataFrame(rows, columns=col_names)
parent_df

,AGE,SEX
0,19,Female
1,19,Male
2,20,Female
3,20,Male
4,21,Female


In [25]:
# # Inspecting our social media table

# cur = connection.cursor()
# for row in cur.execute("SELECT * FROM social_media_table WHERE ROWNUM <= 5"):
#     print(row)
# cur.close()

In [27]:
# Inspecting our social media table
cur = connection.cursor()
cur.execute("SELECT * FROM social_media_table WHERE ROWNUM <= 5")
rows = cur.fetchall()  # fetch all rows as a list of tuples

# Extract column names from the cursor description
col_names = [desc[0] for desc in cur.description]
cur.close()

# Create a DataFrame with the rows and the column names
social_media_df = pd.DataFrame(rows, columns=col_names)
social_media_df

,UNIQUE_ID,AGE,SEX,HAVE_SNS,USUAL_SNS,SNS_DEVICE,INTERNET,SNS_USE_LENGTH,POST_FREQ,SNS_USE_DAILY,SNS_USE_CASE,SNS_FRIENDS,SNS_PERSONAL,SNS_TAGS,SNS_CONTENT,SNS_GOOD,SNS_BELIEVE,PEER_PRESSURE,SNS_INFLUENCE,SNS_WELLBEING
0,1,26,Female,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),5-10 years,Less than 1 per day,1-3 hours,Frequently at anytime,Less than 500,All of them,Less than 5,To stay connected with people,Yes,No,No,Sometimes,No
1,2,24,Female,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),5-10 years,3-5 per day,More than 5 hours,Frequently at anytime,500-2000,All of them,More than 20,To stay connected with people,Yes,No,No,Sometimes,Yes
2,3,23,Female,Yes,Twitter,Mobile Phone,Broadband (Wi-Fi),Less than 2-year,More than 5 per day,More than 5 hours,Night to late night,2000-4000,Most of them,10-20,For time pass,Yes,Yes,Yes,Sometimes,Yes
3,4,27,Female,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),5-10 years,Less than 1 per day,1-3 hours,Frequently at anytime,500-2000,Most of them,More than 20,For time pass,Yes,No,Yes,Sometimes,No
4,5,19,Male,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),2-5 years,Less than 1 per day,3-5 hours,Night to late night,500-2000,Many of them,More than 20,To make new friends,Yes,Yes,No,Not at all,No


In [28]:
# # Inspecting our mental health table

# cur = connection.cursor()
# for row in cur.execute("SELECT * FROM mental_health_table WHERE ROWNUM <= 5"):
#     print(row)
# cur.close()

In [29]:
# Inspecting our social media table
cur = connection.cursor()
cur.execute("SELECT * FROM mental_health_table WHERE ROWNUM <= 5")
rows = cur.fetchall()  # fetch all rows as a list of tuples

# Extract column names from the cursor description
col_names = [desc[0] for desc in cur.description]
cur.close()

# Create a DataFrame with the rows and the column names
mental_health_df = pd.DataFrame(rows, columns=col_names)
mental_health_df

,UNIQUE_ID,AGE,SEX,GENERAL_HEALTH,PHQ_SCORE,PHQ_RESULT,GAD7_SCORE,GAD7_RESULT,HOPELESS,OVERWHELMED,...,THERAPY,DEPRESSION_MEDICATION,ACHA_12MONTHS_ANY_COMP,STUDENT,RACE_1,RACE_2,RACE_3,RACE_4,RACE_5,RACE_6
0,317,20,Male,Good,11,Moderate depression,1,minimal anxiety,1-2 times,1-2 times,...,No,No,4,Yes,0,0,1,0,0,0
1,318,21,Male,Good,13,Moderate depression,4,minimal anxiety,1-2 times,11 or more times,...,No,No,0,Yes,0,0,0,1,0,0
2,319,23,Male,Good,9,Mild depression,3,minimal anxiety,5-6 times,11 or more times,...,No,No,2,Yes,0,0,1,0,0,0
3,320,30,Male,Fair,17,Moderately severe depression,6,mild anxiety,Never,3-4 times,...,No,No,0,Yes,1,0,0,0,0,0
4,321,20,Male,Fair,9,Mild depression,4,minimal anxiety,5-6 times,3-4 times,...,No,No,1,Yes,1,0,0,0,0,0


## Conducting Data Analysis

In [40]:
mh_query1 = """
SELECT age, 
    COUNT(unique_id) AS sm_count,
    AVG(phq_score) AS avg_phq
FROM mental_health_table
GROUP BY age
ORDER BY avg_phq DESC
"""

cur = connection.cursor()
cur.execute(mh_query1)

# Fetch rows and columns
rows = cur.fetchall()
col_names = [desc[0] for desc in cur.description]
cur.close()

# Create a DataFrame using the fetched data
mh_query1_df = pd.DataFrame(rows, columns=col_names)
mh_query1_df.head()

,AGE,SM_COUNT,AVG_PHQ
0,32,3,19.0
1,42,1,15.0
2,38,5,13.6
3,33,5,13.2
4,30,6,12.5


In [ ]:
sm_query1 = """
SELECT age, 
    COUNT(unique_id) AS sm_count,
    AVG(phq_score) AS avg_phq
FROM social_media_table
GROUP BY age
ORDER BY avg_phq DESC
"""

cur = connection.cursor()
cur.execute(sm_query1)

# Fetch rows and columns
rows = cur.fetchall()
col_names = [desc[0] for desc in cur.description]
cur.close()

# Create a DataFrame using the fetched data
sm_query1_df = pd.DataFrame(rows, columns=col_names)
sm_query1_df.head()

In [28]:
def sql_preprocessing(df, id_col='unique_id'):
    """
    preprocesses the data frame by creating a unique identifier column with sequential numbers, 
    converting 'age' to numeric and 'sex' to string, and
    reordering columns so that the unique identifier, 'age', and 'sex' appear first.
    """
    # Create a unique identifier column
    df = df.reset_index(drop=True)
    df[id_col] = df.index + 1

    # Ensure consistent data types for the keys
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['sex'] = df['sex'].astype(str)
    
    # Reorder the columns with unique_id, age, and sex first
    ordered_cols = [id_col, 'age', 'sex'] + [col for col in df.columns if col not in [id_col, 'age', 'sex']]
    df = df[ordered_cols]
    
    return df

sm_df_cleaned = sql_preprocessing(sm_df_cleaned)
mh_df_cleaned = sql_preprocessing(mh_df_cleaned)

In [29]:
sm_df_cleaned.head()

,unique_id,age,sex,have_sns,usual_sns,sns_device,internet,sns_use_length,post_freq,sns_use_daily,sns_use_case,sns_friends,sns_personal,sns_tags,sns_content,sns_good,sns_believe,peer_pressure,sns_influence,sns_wellbeing
0,1,26.0,Female,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),5-10 years,Less than 1 per day,1-3 hours,Frequently at anytime,Less than 500,All of them,Less than 5,To stay connected with people,Yes,No,No,Sometimes,No
1,2,24.0,Female,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),5-10 years,3-5 per day,More than 5 hours,Frequently at anytime,500-2000,All of them,More than 20,To stay connected with people,Yes,No,No,Sometimes,Yes
2,3,23.0,Female,Yes,Twitter,Mobile Phone,Broadband (Wi-Fi),Less than 2-year,More than 5 per day,More than 5 hours,Night to late night,2000-4000,Most of them,10-20,For time pass,Yes,Yes,Yes,Sometimes,Yes
3,4,27.0,Female,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),5-10 years,Less than 1 per day,1-3 hours,Frequently at anytime,500-2000,Most of them,More than 20,For time pass,Yes,No,Yes,Sometimes,No
4,5,19.0,Male,Yes,Facebook,Mobile Phone,Broadband (Wi-Fi),2-5 years,Less than 1 per day,3-5 hours,Night to late night,500-2000,Many of them,More than 20,To make new friends,Yes,Yes,No,Not at all,No


In [30]:
mh_df_cleaned.head()

,unique_id,age,sex,general_health,phq_score,phq_result,gad7_score,gad7_result,hopeless,overwhelmed,...,therapy,depression_medication,acha_12months_any_comp,student,race_1,race_2,race_3,race_4,race_5,race_6
0,1,22,Female,Very Good,6,Mild depression,0,minimal anxiety,Never,11 or more times,...,No,No,2,Yes,0,0,0,1,0,0
1,2,21,Female,Good,19,Moderately severe depression,4,minimal anxiety,1-2 times,5-6 times,...,No,No,0,Yes,0,0,0,1,0,0
2,3,23,Female,Good,10,Moderate depression,9,mild anxiety,7-8 times,11 or more times,...,No,No,2,Yes,0,0,1,1,0,0
3,4,24,Female,Good,11,Moderate depression,10,moderate anxiety,9-10 times,9-10 times,...,No,No,2,Yes,1,0,0,0,0,0
4,5,22,Female,Very Good,6,Mild depression,18,severe anxiety,7-8 times,3-4 times,...,No,No,5,Yes,1,0,0,0,0,0


In [31]:
# Retrieving unique combinations of age and sex in both data frames (to be used as PKs for parent table)
unique_sm = sm_df_cleaned[['age', 'sex']].drop_duplicates().dropna()
unique_mh = mh_df_cleaned[['age', 'sex']].drop_duplicates().dropna()
parent_df = unique_sm.merge(unique_mh, on=['age', 'sex'], how='inner')

parent_df.head()

,age,sex
0,26.0,Female
1,24.0,Female
2,23.0,Female
3,27.0,Female
4,19.0,Male


In [39]:
# Defining functions to map our Python dataframes to Oracle
# CREATE TABLE
# Generates the create table statement from the data frame, including constraints
def generate_create_table_statement(df, table_name, primary_keys=None, 
                                    foreign_keys=None, not_null_columns=None):
    columns_defs = []
    for col in df.columns:
        dtype = df[col].dtype
        oracle_type = map_dtype_to_oracle(dtype, df[col])
        col_def = f"{col} {oracle_type}"
        if not_null_columns and col in not_null_columns:
            col_def += " NOT NULL"
        columns_defs.append(col_def)
    
    create_stmt = f"CREATE TABLE {table_name} (\n    " + ",\n    ".join(columns_defs)
    # Key constraints (PK, FK, NOT NULL)
    if primary_keys:
        pk = ", ".join(primary_keys)
        create_stmt += f",\n    PRIMARY KEY ({pk})"
    if foreign_keys:
        for fk in foreign_keys:
            child_cols, ref_table, parent_cols = fk
            fk_cols_str = ", ".join(child_cols)
            parent_cols_str = ", ".join(parent_cols)
            create_stmt += f",\n    FOREIGN KEY ({fk_cols_str}) REFERENCES {ref_table} ({parent_cols_str})"
    
    create_stmt += "\n);"
    return create_stmt
    
# Maps a pandas data type to its corresponding Oracle SQL type
def map_dtype_to_oracle(dtype, col_data):
    """
    mapping the data frame data types to Oracle equivalents
    """
    if pd.api.types.is_integer_dtype(dtype):
        return "NUMBER"
    elif pd.api.types.is_float_dtype(dtype):
        return "NUMBER"
    elif pd.api.types.is_object_dtype(dtype):
        max_len = col_data.astype(str).map(len).max()
        max_len = max(max_len, 1)
        return f"VARCHAR2({max_len})"
    else:
        return "VARCHAR2(255)"

In [37]:
# INSERT INTO
# Generate the insert statements for every row
def generate_insert_statements(df, table_name):
    inserts = []
    columns = df.columns.tolist()
    for idx, row in df.iterrows():
        values = [format_value_for_sql(row[col]) for col in columns]
        col_list = ", ".join(columns)
        val_list = ", ".join(values)
        insert_stmt = f"INSERT INTO {table_name} ({col_list}) VALUES ({val_list});"
        inserts.append(insert_stmt)
    return inserts
    
# Formats the individual instance to ensure it can be mapped
def format_value_for_sql(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, (int, float)):
        return str(value)
    value = str(value).replace("'", "''")
    return f"'{value}'"

In [38]:
# Defining variables and executing the functions to map the dataframes to Oracle

# Define table names
parent_table_name = "parent_table"
social_media_table_name = "social_media_table"
mental_health_table_name = "mental_health_table"
# Define primary keys
parent_pk = ['age', 'sex']  
social_media_pk = ['unique_id']
mental_health_pk = ['unique_id']
parent_not_null = ['age', 'sex']
child_not_null = ['unique_id', 'age', 'sex']
child_fk = [ (["age", "sex"], parent_table_name, ["age", "sex"]) ]

# Executing create table function
parent_create_sql = generate_create_table_statement(
    parent_df, parent_table_name, 
    primary_keys=parent_pk, 
    not_null_columns=parent_not_null,
)
social_media_create_sql = generate_create_table_statement(
    sm_df_cleaned, social_media_table_name, 
    primary_keys=social_media_pk, 
    foreign_keys=child_fk, 
    not_null_columns=child_not_null
)
mental_health_create_sql = generate_create_table_statement(
    mh_df_cleaned, mental_health_table_name, 
    primary_keys=mental_health_pk, 
    foreign_keys=child_fk, 
    not_null_columns=child_not_null
)

# Executing insert into function
parent_inserts = generate_insert_statements(parent_df, parent_table_name)
social_media_inserts = generate_insert_statements(sm_df_cleaned, social_media_table_name)
mental_health_inserts = generate_insert_statements(mh_df_cleaned, mental_health_table_name)

In [40]:
# Write everything to a SQL file

with open("sm_mh_schema.sql", "w") as f:
    f.write("-- Parent Table\n")
    f.write(parent_create_sql + "\n\n")
    for stmt in parent_inserts:
        f.write(stmt + "\n")
    
    f.write("\n-- Social Media Table\n")
    f.write(social_media_create_sql + "\n\n")
    for stmt in social_media_inserts:
        f.write(stmt + "\n")
    
    f.write("\n-- Mental Health Table\n")
    f.write(mental_health_create_sql + "\n\n")
    for stmt in mental_health_inserts:
        f.write(stmt + "\n")

print("sql schema file created successfully.")

sql schema file created successfully.
